# Phase 5 — Train baseline (SlowFast MLP) and fusion (VideoMAE+CLIP) classifiers

Both share the same training loop (`training/loop.py`): class-weighted cross-entropy (or `WeightedRandomSampler`, configurable), early stopping on validation macro-F1, best-model checkpointing. The baseline establishes a reference number before judging whether the frozen-encoder fusion model is worth its extra extraction cost.

In [2]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parents[1]  # notebooks/ -> project/ -> repo root
sys.path.insert(0, str(REPO_ROOT))

from project.config import load_config, resolve_path

cfg = load_config()


## Baseline: SlowFast (2304-d) → MLP

In [3]:
from project.datasets.feature_dataset import CachedEmbeddingDataset
from project.models.baseline_mlp import SlowFastBaselineMLP
from project.models.encoders import get_device
from project.training.loop import train_classifier

device = get_device(cfg['encoders']['device_preference'])
slowfast_dir = resolve_path(cfg, 'paths.features_dir') / 'slowfast'

train_ds = CachedEmbeddingDataset(resolve_path(cfg, 'paths.train_csv'), [(slowfast_dir, 'parent_video_id')])
val_ds = CachedEmbeddingDataset(resolve_path(cfg, 'paths.val_csv'), [(slowfast_dir, 'parent_video_id')])
print('train:', len(train_ds), ' val:', len(val_ds))

baseline_model = SlowFastBaselineMLP(input_dim=2304, hidden_dim=cfg['training']['baseline_hidden_dim'],
                                      num_classes=len(cfg['emotion_to_id']), dropout=cfg['training']['baseline_dropout'])
baseline_ckpt = resolve_path(cfg, 'paths.checkpoints_dir') / 'baseline_slowfast_mlp_best.pt'
baseline_history = train_classifier(baseline_model, train_ds, val_ds, cfg, baseline_ckpt, device, model_name='baseline-slowfast')

train: 2232  val: 479
[baseline-slowfast] epoch 001  train_loss=1.7044  val_loss=1.5136  val_acc=0.3048  val_macro_f1=0.2422
[baseline-slowfast] epoch 002  train_loss=1.5505  val_loss=1.5025  val_acc=0.2965  val_macro_f1=0.2043
[baseline-slowfast] epoch 003  train_loss=1.5333  val_loss=1.4923  val_acc=0.3382  val_macro_f1=0.2257
[baseline-slowfast] epoch 004  train_loss=1.5175  val_loss=1.4594  val_acc=0.4071  val_macro_f1=0.3246
[baseline-slowfast] epoch 005  train_loss=1.4835  val_loss=1.4090  val_acc=0.4614  val_macro_f1=0.3830
[baseline-slowfast] epoch 006  train_loss=1.4673  val_loss=1.3887  val_acc=0.4551  val_macro_f1=0.3852
[baseline-slowfast] epoch 007  train_loss=1.4425  val_loss=1.3701  val_acc=0.4760  val_macro_f1=0.4105
[baseline-slowfast] epoch 008  train_loss=1.4343  val_loss=1.3692  val_acc=0.4614  val_macro_f1=0.3957
[baseline-slowfast] epoch 009  train_loss=1.3963  val_loss=1.3512  val_acc=0.5136  val_macro_f1=0.4351
[baseline-slowfast] epoch 010  train_loss=1.4195  v

## Fusion: VideoMAE (768-d) + CLIP (512-d) → MLP

In [4]:
from project.models.encoders import FrozenCLIPEncoder, FrozenVideoMAEEncoder
from project.models.fusion_classifier import FusionClassifier

videomae_dir = resolve_path(cfg, 'paths.videomae_features_dir')
clip_dir = resolve_path(cfg, 'paths.clip_features_dir')
feature_specs = [(videomae_dir, 'video_id'), (clip_dir, 'video_id')]

train_ds_fusion = CachedEmbeddingDataset(resolve_path(cfg, 'paths.train_csv'), feature_specs)
val_ds_fusion = CachedEmbeddingDataset(resolve_path(cfg, 'paths.val_csv'), feature_specs)
print('train:', len(train_ds_fusion), ' val:', len(val_ds_fusion))

fusion_model = FusionClassifier(videomae_dim=FrozenVideoMAEEncoder.EMBED_DIM, clip_dim=FrozenCLIPEncoder.EMBED_DIM,
                                 hidden_dim=cfg['training']['fusion_hidden_dim'], num_classes=len(cfg['emotion_to_id']),
                                 dropout=cfg['training']['fusion_dropout'])
fusion_ckpt = resolve_path(cfg, 'paths.checkpoints_dir') / 'fusion_classifier_best.pt'
fusion_history = train_classifier(fusion_model, train_ds_fusion, val_ds_fusion, cfg, fusion_ckpt, device, model_name='fusion-videomae-clip')

train: 2232  val: 479
[fusion-videomae-clip] epoch 001  train_loss=1.1051  val_loss=0.7555  val_acc=0.7641  val_macro_f1=0.7530
[fusion-videomae-clip] epoch 002  train_loss=0.6351  val_loss=0.6575  val_acc=0.7766  val_macro_f1=0.7654
[fusion-videomae-clip] epoch 003  train_loss=0.5539  val_loss=0.6418  val_acc=0.7933  val_macro_f1=0.7838
[fusion-videomae-clip] epoch 004  train_loss=0.4899  val_loss=0.5958  val_acc=0.8058  val_macro_f1=0.7959
[fusion-videomae-clip] epoch 005  train_loss=0.4640  val_loss=0.5843  val_acc=0.8079  val_macro_f1=0.7985
[fusion-videomae-clip] epoch 006  train_loss=0.4353  val_loss=0.6197  val_acc=0.7933  val_macro_f1=0.7826
[fusion-videomae-clip] epoch 007  train_loss=0.4205  val_loss=0.5917  val_acc=0.7996  val_macro_f1=0.7883
[fusion-videomae-clip] epoch 008  train_loss=0.3887  val_loss=0.6046  val_acc=0.8163  val_macro_f1=0.8065
[fusion-videomae-clip] epoch 009  train_loss=0.3840  val_loss=0.6158  val_acc=0.8038  val_macro_f1=0.7935
[fusion-videomae-clip] e